# Pokémon TCG team arena

A shared match runner for the whole team. Fill in the forms and run them from top to bottom, or select **Runtime → Run all** in Colab. The defaults run Random against Priority without a model checkpoint.

For a teammate's model, select **Python script** and enter a `.py` file or agent folder. The script owns its model architecture, preprocessing, weight loading, and move selection. The arena calls its exported policy and scores the games.

Each series is first to three wins, with alternating seats and fresh native deals. Draws are replayed. The [Deep Sets walkthrough](deep_sets_walkthrough.ipynb) remains separate.


**Colab: choose Runtime → Run all.** Setup clones the public [project repository](https://github.com/le21-j/pokemon-tcg-fpga) and installs dependencies automatically.

In [ ]:
# @title 1. Automatic setup {display-mode: "form"}
# @markdown Run All clones the public project and installs dependencies automatically in Colab. Optional overrides can stay at their defaults.
REPOSITORY_PATH = "" # @param {type:"string"}
MOUNT_GOOGLE_DRIVE = False # @param {type:"boolean"}
INSTALL_LOCALLY = False # @param {type:"boolean"}

"""Standard-library Colab setup for the public Pokémon TCG project repository."""

import os
from pathlib import Path
import subprocess
import sys
import tempfile


PUBLIC_REPOSITORY = "https://github.com/le21-j/pokemon-tcg-fpga.git"


def running_in_colab():
    if "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ:
        return True
    try:
        import google.colab
        return True
    except ImportError:
        return False


def _project_ready(root):
    return all((root / name).is_file() for name in (
        "project/arena.py", "project/sample_sdk.py", "project/agent/model.py",
        "project/rl_baseline/train.py", "project/agent_template/agent.py",
        "project/requirements-colab.txt"))


def setup_project(repository_path="", *, repository_url=PUBLIC_REPOSITORY, repository_ref="main",
                  in_colab=None, workspace_parent=None, install_locally=False):
    in_colab = running_in_colab() if in_colab is None else in_colab
    if repository_path.strip():
        root = Path(repository_path).expanduser().resolve()
        if not _project_ready(root):
            raise ValueError("REPOSITORY_PATH must contain the Pokémon TCG project")
    elif not in_colab:
        root = next((p for p in [Path.cwd(), *Path.cwd().parents] if _project_ready(p)), None)
        if root is None:
            raise ValueError("Run inside the project checkout or set REPOSITORY_PATH")
    else:
        parent = Path(workspace_parent) if workspace_parent is not None else (
            Path("/content") if Path("/content").is_dir() else Path.cwd())
        parent.mkdir(parents=True, exist_ok=True)
        name = "pokemon-tcg-fpga"
        root = parent / name
        retry = 0
        while root.exists() and not _project_ready(root):
            retry += 1
            root = parent / f"{name}-{retry}"
        if not root.exists():
            with tempfile.TemporaryDirectory(prefix="tcg-clone-", dir=parent) as temporary:
                checkout = Path(temporary) / "checkout"
                subprocess.run(["git", "clone", "--depth", "1", "--branch", repository_ref,
                                repository_url, str(checkout)],
                               env=dict(os.environ, GIT_TERMINAL_PROMPT="0"), check=True)
                if not _project_ready(checkout):
                    raise RuntimeError("The downloaded repository is missing project files")
                checkout.rename(root)
            print("Cloned the public Pokémon TCG project")
    if in_colab or install_locally:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
                               "-r", str(root / "project/requirements-colab.txt")])
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))
    print("Project ready:", root)
    return root


if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

ROOT = setup_project(REPOSITORY_PATH, install_locally=INSTALL_LOCALLY)

# --- End automatic project setup ---

import hashlib
import json
from collections import Counter
import pandas as pd
from IPython.display import display
from project.collect_matches import ENGINE_ROOT, download_runtime, load_runtime, runtime_identity, choose_random
from project.arena import Agent, ValueSearchAgent, load_deck_pool, validate_deck_pool, priority_action, run_series
from project.agent.native_search import NativeSearch
from project.script_agent import ScriptAgent

print("Workspace:", ROOT)
print("Notebook Python:", sys.executable)

In [ ]:
# @title 2a. Configure agent A {display-mode: "form"}
# @markdown **Python script:** path to a file or folder containing the teammate's inference code and weights. **Deep Sets checkpoint:** path to this project's `.pt` checkpoint. Random/Priority need no path.
AGENT_A_NAME = "Team A" # @param {type:"string"}
AGENT_A_SOURCE = "Random" # @param ["Python script", "Random", "Priority", "Deep Sets checkpoint"]
AGENT_A_PATH = "" # @param {type:"string"}
AGENT_A_DECK = "random" # @param ["random", "sample-water", "mega-lucario", "mega-abomasnow", "dragapult", "JSON file"]
# @markdown For **JSON file**, supply a JSON list of 60 card IDs. Paths can be absolute or relative to the repository root.
AGENT_A_DECK_FILE = "" # @param {type:"string"}

In [ ]:
# @title 2b. Configure agent B {display-mode: "form"}
# @markdown **Python script:** path to a file or folder containing the teammate's inference code and weights. **Deep Sets checkpoint:** path to this project's `.pt` checkpoint. Random/Priority need no path.
AGENT_B_NAME = "Team B" # @param {type:"string"}
AGENT_B_SOURCE = "Priority" # @param ["Python script", "Random", "Priority", "Deep Sets checkpoint"]
AGENT_B_PATH = "" # @param {type:"string"}
AGENT_B_DECK = "random" # @param ["random", "sample-water", "mega-lucario", "mega-abomasnow", "dragapult", "JSON file"]
# @markdown For **JSON file**, supply a JSON list of 60 card IDs. Paths can be absolute or relative to the repository root.
AGENT_B_DECK_FILE = "" # @param {type:"string"}

In [ ]:
# @title 3. Script integration options {display-mode: "form"}
# @markdown Defaults expect `agent(observation) -> list[int]`. Change the exported function name or signature to match a teammate's script. A directory resolves to `agent.py`, then `main.py`, then `__init__.py`, or its only `.py` file.
AGENT_A_ENTRYPOINT = "agent" # @param {type:"string"}
AGENT_B_ENTRYPOINT = "agent" # @param {type:"string"}
AGENT_A_CALL_STYLE = "observation" # @param ["observation", "observation_rng", "observation_configuration"]
AGENT_B_CALL_STYLE = "observation" # @param ["observation", "observation_rng", "observation_configuration"]
# @markdown Optional Python executable for each script, such as `/path/to/their-env/bin/python`. Blank uses the notebook's Python. Install the teammate's dependencies in that environment.
AGENT_A_PYTHON = "" # @param {type:"string"}
AGENT_B_PYTHON = "" # @param {type:"string"}
# @markdown For the `observation_configuration` signature, these JSON objects are passed as the second argument. Otherwise they are unused.
AGENT_A_CONFIGURATION = "{}" # @param {type:"string"}
AGENT_B_CONFIGURATION = "{}" # @param {type:"string"}
# @markdown Applies separately to startup/weight loading and to each response. A timeout stops the series and reports an error; it does not invent a match outcome.
SCRIPT_TIMEOUT_SECONDS = 60 # @param {type:"number"}

In [ ]:
# @title 4. Match settings {display-mode: "form"}
# @markdown Decks stay fixed within a series unless reselection is enabled. The seed controls Python/deck choices and the supplied RNG for `observation_rng` policies. Native shuffling has no exposed seed.
RESELECT_DECKS_EACH_GAME = False # @param {type:"boolean"}
SERIES_SEED = 17 # @param {type:"integer"}
MAX_TOTAL_GAMES = 20 # @param {type:"integer"}
MAX_STEPS_PER_GAME = 10000 # @param {type:"integer"}
DOWNLOAD_RUNTIME_IF_MISSING = True # @param {type:"boolean"}
RESULTS_DIRECTORY = "project/results/arena" # @param {type:"string"}

In [ ]:
# @title 5. Load agents and validate decks {display-mode: "form"}

def setting_path(value, label):
    if not value.strip():
        raise ValueError(f"Enter {label} in the agent form")
    path = Path(value).expanduser()
    return path.resolve() if path.is_absolute() else (ROOT / path).resolve()

if not (ENGINE_ROOT / "cg/game.py").exists():
    if not DOWNLOAD_RUNTIME_IF_MISSING:
        raise FileNotFoundError("Prepare the CABT runtime or enable its download in match settings")
    download_runtime(ENGINE_ROOT)
game_api, sim = load_runtime(ENGINE_ROOT)
search_api = NativeSearch(sim.Battle.lib)
DECK_POOL = load_deck_pool()


def selected_deck(name, choice, filename):
    if choice != "JSON file":
        return choice
    path = setting_path(filename, f"{name}'s deck JSON file")
    cards = json.loads(path.read_text())
    DECK_POOL[f"custom-{name}"] = cards
    return f"custom-{name}"


deck_a = selected_deck(AGENT_A_NAME, AGENT_A_DECK, AGENT_A_DECK_FILE)
deck_b = selected_deck(AGENT_B_NAME, AGENT_B_DECK, AGENT_B_DECK_FILE)
validate_deck_pool(game_api, sim, DECK_POOL)


def make_agent(name, source, path, deck, entrypoint, call_style, python, configuration):
    if source == "Python script":
        action = ScriptAgent(setting_path(path, f"{name}'s agent script/folder"),
                             entrypoint=entrypoint, call_style=call_style,
                             python_executable=python.strip() or None,
                             timeout=SCRIPT_TIMEOUT_SECONDS,
                             configuration=json.loads(configuration))
    elif source == "Random":
        action = choose_random
    elif source == "Priority":
        action = priority_action
    elif source == "Deep Sets checkpoint":
        action = ValueSearchAgent.from_checkpoint(
            setting_path(path, f"{name}'s checkpoint"), search_api, list(DECK_POOL.values()))
    else:
        raise ValueError(f"Unknown source: {source}")
    return Agent(name, action, deck)


agent_a = make_agent(AGENT_A_NAME, AGENT_A_SOURCE, AGENT_A_PATH, deck_a,
                     AGENT_A_ENTRYPOINT, AGENT_A_CALL_STYLE, AGENT_A_PYTHON, AGENT_A_CONFIGURATION)
agent_b = make_agent(AGENT_B_NAME, AGENT_B_SOURCE, AGENT_B_PATH, deck_b,
                     AGENT_B_ENTRYPOINT, AGENT_B_CALL_STYLE, AGENT_B_PYTHON, AGENT_B_CONFIGURATION)
display(pd.DataFrame([
    {"agent": agent.name, "source": source, "entry file / checkpoint": path or "built-in", "deck": agent.deck}
    for agent, source, path in ((agent_a, AGENT_A_SOURCE, AGENT_A_PATH), (agent_b, AGENT_B_SOURCE, AGENT_B_PATH))
]))
display(pd.DataFrame([{"deck": name, "cards": len(cards)} for name, cards in DECK_POOL.items()]))

In [ ]:
# @title 6. Play the first-to-three series {display-mode: "form"}
# @markdown A fresh external Python worker starts for each agent at the beginning of each game. It stays alive across that game's moves, so the script can load its weights once and keep its own state.
try:
    series = run_series(
        game_api, sim, agent_a, agent_b, DECK_POOL,
        seed=SERIES_SEED,
        reselect_decks_each_game=RESELECT_DECKS_EACH_GAME,
        max_total_games=MAX_TOTAL_GAMES,
        max_steps=MAX_STEPS_PER_GAME,
    )
finally:
    for competitor in (agent_a, agent_b):
        close = getattr(competitor.action, "close", None)
        if close is not None:
            close()
assert sim.Battle.battle_ptr is None
assert not series["completed"] or max(series["wins"]) == 3

In [ ]:
# @title 7. Results and agent counters {display-mode: "form"}
rows = []
for result in series["games"]:
    fingerprint = hashlib.sha256(json.dumps(result["opening_hands"], sort_keys=True).encode()).hexdigest()[:12]
    rows.append({
        "game": result["game"], "winner": result["winner"],
        f"{agent_a.name} wins": result["score"][0],
        f"{agent_b.name} wins": result["score"][1],
        f"{agent_a.name} deck": result["decks"][agent_a.name]["name"],
        f"{agent_b.name} deck": result["decks"][agent_b.name]["name"],
        "seat 0": result["seat0"], "decisions": result["decisions"],
        "seconds": round(result["seconds"], 3), "first-hand fingerprint": fingerprint,
    })
display(pd.DataFrame(rows))
print("Winner:", series["winner"] if series["completed"] else "Incomplete: total-game guard reached")
print("Score:", dict(zip(series["agents"], series["wins"])))
print("Draws replayed:", series["draws"])

totals = {agent.name: Counter() for agent in (agent_a, agent_b)}
for result in series["games"]:
    for name, stats in result["agent_stats"].items():
        totals[name].update(stats)
stats_table = pd.DataFrame([{"agent": name, **stats} for name, stats in totals.items()]).fillna(0)
if "script_decisions" in stats_table:
    stats_table["script mean response ms"] = [
        1000 * stats["script_action_seconds"] / stats["script_decisions"] if stats["script_decisions"] else None
        for stats in totals.values()
    ]
display(stats_table)

In [ ]:
# @title 8. Save the series report {display-mode: "form"}

def competitor_identity(agent, source, path):
    identity = {"name": agent.name, "source": source}
    if isinstance(agent.action, ScriptAgent):
        identity.update(agent.action.identity())
    elif source == "Deep Sets checkpoint":
        checkpoint = setting_path(path, "checkpoint")
        identity.update(checkpoint=str(checkpoint), checkpoint_sha256=hashlib.sha256(checkpoint.read_bytes()).hexdigest())
    return identity


series["runtime"] = runtime_identity(ENGINE_ROOT)
series["competitors"] = [
    competitor_identity(agent_a, AGENT_A_SOURCE, AGENT_A_PATH),
    competitor_identity(agent_b, AGENT_B_SOURCE, AGENT_B_PATH),
]
output = setting_path(RESULTS_DIRECTORY, "results directory") / f"{series['series_id']}.json"
output.parent.mkdir(parents=True, exist_ok=True)
output.write_text(json.dumps(series, indent=2) + "\n")
print("Saved results:", output)

## What a teammate supplies

Pass along an inference script and any files it needs, including weights and local modules. The arena does not need to know whether the model is Deep Sets, a transformer, a tree, or another architecture. A standalone training script or command-line program needs a small policy wrapper that exposes move selection.

A minimal policy interface is:

```python
# agent.py
# Load your trained model and weights here, once per game.
def agent(observation):
    selection = observation["select"]
    return list(range(selection["minCount"]))  # Legal baseline; replace with your model.
```

Each worker uses the script's folder as its working directory, supports sibling imports, and accepts package-relative imports such as `from .model import ...`. A folder with several possible entry files should be supplied as an explicit `.py` path. For a folder path, `agent.py` takes precedence over `main.py` and `__init__.py`.

Choose `observation_rng` for `agent(observation, rng)`, or `observation_configuration` for `agent(observation, configuration)`, where configuration is the JSON object from the form. The returned action must be a Python list of legal integer option indices. An optional module or callable hook `start_game(own_deck)` receives only that agent's own deck. Callable instances work; export an instance rather than an uninstantiated class.

Python/native stdout is captured so model-loading messages do not break the response protocol. Import errors, action errors, and response timeouts stop the run and report the script failure. The opponent's selected deck is outside the callback interface. Scripts manage any RNGs they create themselves.

Reports include deck lists, score progression, seats, first-observed hands, decision counts, elapsed time, optional counters, runtime hashes, and entry-script or built-in checkpoint hashes. Entry-script hashes do not cover separately loaded weights; include the model version in each competitor's name. Script response timings include IPC, preprocessing, policy/search, and inference, and exclude worker startup and import-time weight loading. Use several series and consistent decks/resource budgets for performance estimates.

Built-in Deep Sets agents use one-step native search and display counted Priority fallbacks for unsupported selection/belief contexts.

## References

- [Notebook form layout reference: LibreLane](https://colab.research.google.com/github/librelane/librelane/blob/main/notebook.ipynb)
- [CABT API](https://matsuoinstitute.github.io/cabt/)
- [Pinned native game runtime](https://github.com/Kaggle/kaggle-environments/tree/998b15c14b1b77aafb1843bed63858455035affd/kaggle_environments/envs/cabt/cg)
- [Original native search wrapper, cg-lib v9](https://www.kaggle.com/datasets/kiyotah/cg-lib/versions/9)
- Example decks from Kiyota: [Mega Lucario](https://www.kaggle.com/datasets/kiyotah/mega-lucario-ex-deck), [Mega Abomasnow](https://www.kaggle.com/datasets/kiyotah/mega-abomasnow-ex-deck), [Dragapult](https://www.kaggle.com/datasets/kiyotah/dragapult-ex-deck)